# CLIP 은 뭘 보고 있나 — 색 지우기 / 내용 부수기 실험

오늘 얘기하다가 나온 관찰이 계속 걸렸다: **"임베딩이 그림체가 아니라 사물을 식별하는 경향이 있는 것 같다."**
느낌으로는 다들 동의했는데, 이게 진짜인지 숫자로 확인해보고 싶어서 가설 두 개를 세워 돌려봤다.

| 가설 | 실험 | 예상 |
|---|---|---|
| 색이 사물·분위기 단서라면, 색을 지우면 그림체가 더 보일 것 | 흑백으로 바꿔서 CLIP | 점수가 오른다? |
| CLIP 이 내용을 보고 있다면, 내용을 부수면 그림체만 남을 것 | 그림을 4x4 조각으로 섞어서 CLIP | 점수가 오른다? |

잣대는 팀 리더보드 그대로다: **p@5** (가장 가까운 5장 중 같은 그림체 비율), 무작위 대비 배수,
그리고 **95% 구간** — 두 방법의 차이가 이 구간 폭보다 작으면 이긴 게 아니라고 본다.

먼저 결론부터: **하나는 틀렸고 하나는 맞았다.** 과정은 아래에.


In [ ]:
# 시작 노트북과 같은 준비: 드라이브에서 코퍼스와 채점기를 가져와 푼다
# (런타임 유형을 GPU 로 해두면 채점이 훨씬 빠르다)
from google.colab import drive
drive.mount('/content/drive')

import os, glob, zipfile
CANDS = glob.glob('/content/drive/MyDrive/DLthon_그림체RAG') \
      + glob.glob('/content/drive/MyDrive/*/DLthon_그림체RAG') \
      + glob.glob('/content/drive/MyDrive/*/*/DLthon_그림체RAG') \
      + glob.glob('/content/drive/Shareddrives/*/DLthon_그림체RAG') \
      + glob.glob('/content/drive/.shortcut-targets-by-id/*/DLthon_그림체RAG')
assert CANDS, "공유 폴더 바로가기가 필요하다 (드라이브에서 우클릭 -> 내 드라이브에 바로가기 추가)"

for z in ['daypack_v2.zip', 'kit.zip']:
    with zipfile.ZipFile(os.path.join(CANDS[0], z)) as f:
        f.extractall('/content')
print("준비 완료:", len(glob.glob('/content/daypack_v2/images/*/*.jpg')), "장")


## 가설 1 — 색을 지우면 그림체가 더 보일까?

색은 "무엇이 그려졌나"와 "어떤 분위기냐"를 같이 실어 나르는 단서다. CLIP 이 사물에 끌린다면,
색을 아예 지우고(흑백) 선·명암·질감만 남기면 그림체 인식이 오르지 않을까 생각했다.
예상은: 잉크(원래 무채색)는 오르거나 그대로, 벡터(색이 정체성인 이모지들)는 떨어질 것.

기준선 코드(clip_base.py)에서 **딱 한 줄** — 넣기 전에 흑백 변환 — 만 다르게 했다.


In [ ]:
%%writefile /content/kit/encoders/clip_gray.py
"""실험: 색을 지우면 그림체가 더 잘 보일까?

CLIP 이 사물·색 분위기에 끌린다는 게 우리 관찰이다. 그럼 색 단서를 아예 지우고
(흑백 변환) 넣으면, 남는 건 선·명암·질감이니 그림체 인식이 오르지 않을까 해서 해본다.
예상: 잉크(원래 무채색)는 오르거나 그대로, 벡터(색이 정체성인 이모지들)는 떨어질 것 같다.
"""
import numpy as np
import torch
from PIL import ImageOps
from transformers import CLIPModel, CLIPProcessor

M = "openai/clip-vit-base-patch32"
_dev = "cuda" if torch.cuda.is_available() else "cpu"
_model = CLIPModel.from_pretrained(M).to(_dev).eval()
_proc = CLIPProcessor.from_pretrained(M)


def encode(images, bs=32):
    # 기준선과 다른 곳은 이 한 줄뿐이다: 넣기 전에 흑백으로 바꾼다
    images = [ImageOps.grayscale(im).convert("RGB") for im in images]
    out = []
    with torch.no_grad():
        for i in range(0, len(images), bs):
            b = _proc(images=images[i:i + bs], return_tensors="pt").to(_dev)
            f = _model.get_image_features(**b)
            out.append((f / f.norm(dim=-1, keepdim=True)).cpu().numpy())
    return np.concatenate(out).astype("float32")


In [ ]:
%cd /content/kit
!DAYPACK=/content/daypack_v2 python score.py encoders/clip_gray.py --name "민욱-흑백CLIP" --note "색 단서 제거 실험"


### 결과 1 — 틀렸다. 오히려 떨어졌다

내가 돌렸을 때 **전체 0.3711** (기준선 CLIP 0.3986). 95% 구간이 0.3582~0.3843 이라
기준선과 안 겹친다 — 노이즈가 아니라 진짜 하락이다. (숫자가 크게 다르게 나오면 뭔가 어긋난 것)

해석해보면: **색 자체도 그림체의 일부였다.** 수채화의 번지는 색, 이모지의 쨍한 팔레트 같은 게
그림체 단서였는데 그걸 같이 지워버린 셈이다. "색 = 사물 단서니까 지우면 이득"이라는 내 가설이
너무 단순했다. 색은 사물도 싣고 그림체도 싣는다.


## 가설 2 — 내용을 부수면 그림체만 남을까?

이번엔 색이 아니라 **내용(무엇이 그려졌나)을 직접 부순다.** 그림을 4x4 = 16조각으로 잘라
고정된 순서로 섞으면, 나무·사람·구도는 부서지고 조각 안의 선맛·붓질·팔레트는 남는다.
이걸로 점수가 **오르면**, CLIP 이 그동안 내용을 보고 있었다는 게 숫자로 증명되는 셈이다.

섞는 순서는 seed 로 고정해서 누가 돌려도 같은 결과가 나오게 했다.


In [ ]:
%%writefile /content/kit/encoders/clip_patchshuffle.py
"""실험: 내용(무엇이 그려졌나)을 일부러 부수면 그림체만 남을까?

우리 관찰 = CLIP 은 사물을 식별하는 경향이 있다. 그래서 그림을 4x4 조각으로 잘라
섞어버린다. 나무·사람·구도는 부서지고, 선맛·붓질·팔레트 같은 그림체 단서는 조각 안에 남는다.
이걸로 점수가 오르면 "CLIP 이 그동안 내용을 보고 있었다"가 숫자로 증명되는 셈이다.
섞는 순서는 고정(seed)이라 누가 돌려도 같은 결과가 나온다.
"""
import random
import numpy as np
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

M = "openai/clip-vit-base-patch32"
_dev = "cuda" if torch.cuda.is_available() else "cpu"
_model = CLIPModel.from_pretrained(M).to(_dev).eval()
_proc = CLIPProcessor.from_pretrained(M)

N = 4                                   # 4x4 = 16조각
_order = list(range(N * N))
random.Random(0).shuffle(_order)        # 섞는 순서를 고정 -> 모든 그림에 같은 '파괴'를 가한다


def _scramble(im, size=224):
    # 정사각형으로 맞춘 뒤 조각내서 고정 순서로 다시 붙인다
    im = im.convert("RGB").resize((size, size))
    t = size // N
    tiles = [im.crop((c * t, r * t, (c + 1) * t, (r + 1) * t))
             for r in range(N) for c in range(N)]
    out = Image.new("RGB", (size, size))
    for i, j in enumerate(_order):
        r, c = divmod(i, N)
        out.paste(tiles[j], (c * t, r * t))
    return out


def encode(images, bs=32):
    images = [_scramble(im) for im in images]
    out = []
    with torch.no_grad():
        for i in range(0, len(images), bs):
            b = _proc(images=images[i:i + bs], return_tensors="pt").to(_dev)
            f = _model.get_image_features(**b)
            out.append((f / f.norm(dim=-1, keepdim=True)).cpu().numpy())
    return np.concatenate(out).astype("float32")


In [ ]:
%cd /content/kit
!DAYPACK=/content/daypack_v2 python score.py encoders/clip_patchshuffle.py --name "민욱-패치셔플CLIP" --note "내용 파괴 실험(4x4 조각 섞기)"


### 결과 2 — 맞았다. 내용을 부쉈더니 올랐다

내가 돌렸을 때 **전체 0.4271** (기준선 0.3986). 95% 구간 0.4142~0.4404 가 기준선 구간
(0.3855~0.4128)과 **안 겹친다** — 진짜 상승이다.

그림을 조각내서 "무엇이 그려졌는지"를 못 알아보게 만들었는데 그림체 점수가 **올랐다.**
우리가 느낀 "CLIP 은 사물을 본다"가 맞았다는 뜻이다. 사물 단서를 뺏으니 남은 힘을
질감·선맛 쪽에 쓰게 된 것으로 보인다.

## 정리 — 배운 것과 다음 궁금증

| 방법 | 전체 p@5 | 무작위 대비 |
|---|---|---|
| CLIP 기준선 | 0.3986 | 7.72배 |
| 민욱-흑백CLIP | 0.3711 | 7.19배 |
| 민욱-패치셔플CLIP | 0.4271 | 8.28배 |
| **Gram VGG19 (1위)** | **0.4914** | **9.52배** |

1. **CLIP 이 사물을 본다는 관찰은 맞다** — 내용을 부수니 그림체 점수가 올랐다 (0.3986 -> 0.4271)
2. **색도 그림체 단서다** — 지웠더니 떨어졌다 (0.3986 -> 0.3711). 단서를 "빼는" 방향은 조심해야 한다
3. **그래도 Gram 이 1위다** — Gram 은 애초에 "어디에"를 지우고 질감 통계만 남기는 방법이라던데,
   왜 조각 섞기보다 센지 아직 이해를 못 했다. 다음 궁금증으로 남긴다

한계도 적어둔다: 조각을 섞어도 **조각 안**의 내용(눈, 잎사귀 같은 것)은 남는다. 조각을 더 잘게
하면 내용이 더 부서질 텐데 어디까지 이득인지는 안 해봤다. 그리고 이 숫자들은 daypack_v2 기준이라
코퍼스가 바뀌면 전부 다시 재야 한다 (팀 규칙).
